# DAS version-2 freeze checkpoint

This notebook reads **compact tracked products only**. It does not open raw DAS HDF5, network miniSEED, catalog caches, or any held-out waveform.

**Recorded outcome:** version 2 adds one post-hoc development-tuned rule—at least four of ten blocks at the existing characteristic ratio of 2. It retains 2 of 65 v1 triggers, the two known local development events, and rejects the other 63. This is an implementation/freeze checkpoint, not held-out validation or a catalog-extension result.

In [ ]:
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
matches = [
    path for path in [ROOT, *ROOT.parents]
    if (path / 'outputs' / 'development_das_v2').is_dir()
]
if not matches:
    raise FileNotFoundError('Run from repeaters_v2 or one of its subdirectories')
PROJECT = matches[0]
V1 = PROJECT / 'outputs' / 'development_das'
V2 = PROJECT / 'outputs' / 'development_das_v2'
CONFIG = PROJECT / 'config'
NETWORK_REGISTRATION = PROJECT / 'outputs' / 'heldout_v2' / 'registration'
RUNNER_RELEASE = NETWORK_REGISTRATION / 'network_runner_release.json'
pd.set_option('display.max_columns', 40)
print('Project:', PROJECT)
print('Inputs: compact CSV/JSON/source hashes only; no waveform is opened.')

## 1. Provenance and access ledger

The assertions below fail if the registered config, registration record, v2 candidate table, remotely released runner source, or release artifact changes. Registration and release could inspect metadata and historical-template hashes, but neither opened a held-out waveform, catalog-event row, DAS HDF5, or family label.

In [ ]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

with (CONFIG / 'das_v2_validation.json').open(encoding='utf-8') as handle:
    config = json.load(handle)
with (V2 / 'registration_status.json').open(encoding='utf-8') as handle:
    registration = json.load(handle)
with (V2 / 'status.json').open(encoding='utf-8') as handle:
    status = json.load(handle)
with (NETWORK_REGISTRATION / 'network_registration_status.json').open(encoding='utf-8') as handle:
    network_registration = json.load(handle)
with RUNNER_RELEASE.open(encoding='utf-8') as handle:
    runner_release = json.load(handle)
raw = pd.read_csv(V1 / 'candidate_detections_raw.csv')
v2 = pd.read_csv(V2 / 'candidate_detections.csv')

assert sha256(CONFIG / 'das_v2_validation.json') == status['das_v2_config_sha256']
assert sha256(V2 / 'registration_status.json') == status['registration_status_sha256']
assert sha256(V2 / 'candidate_detections.csv') == status['candidate_table_sha256']
assert registration['heldout_intervals_all_sealed']
assert registration['heldout_total_duration_h'] == 12.0
assert status['v1_raw_candidate_count'] == len(raw) == 65
assert status['v2_retained_candidate_count'] == len(v2) == 2
assert status['development_replay_regression_status'] == 'PASS_DISCLOSED_2_OF_65'
assert network_registration['heldout_intervals_all_sealed_at_registration']
assert network_registration['heldout_total_duration_h'] == 12.0
assert network_registration['heldout_network_waveform_files_opened'] == 0
assert network_registration['heldout_DAS_HDF5_files_opened'] == 0
assert network_registration['heldout_catalog_event_rows_opened'] == 0
assert runner_release['status'] == 'REMOTE_VERIFIED_FOR_HELDOUT_NETWORK_ACCESS'
assert runner_release['runner_commit_sha'] == runner_release['remote_branch_sha']
assert runner_release['remote_branch_sha_verified_equal']
assert runner_release['repository_visibility'] == 'private'
assert runner_release['heldout_network_config_sha256'] == network_registration['heldout_network_config_sha256']
assert runner_release['network_registration_status_sha256'] == sha256(NETWORK_REGISTRATION / 'network_registration_status.json')
assert runner_release['template_input_inventory_sha256'] == sha256(NETWORK_REGISTRATION / 'network_template_input_inventory.csv')
for relative, expected_hash in runner_release['runner_implementation_files'].items():
    assert sha256(PROJECT / relative) == expected_hash

access = {
    'registration: held-out DAS files': registration['heldout_DAS_HDF5_files_opened'],
    'registration: held-out network files': registration['heldout_network_waveform_files_opened'],
    'registration: held-out catalog rows': registration['heldout_catalog_event_rows_opened'],
    'replay: held-out DAS files': status['heldout_DAS_HDF5_files_opened'],
    'replay: held-out network files': status['heldout_network_waveform_files_opened'],
    'replay: network candidate tables': status['network_candidate_tables_opened_during_replay'],
    'replay: catalog event tables': status['catalog_event_time_tables_opened_during_replay'],
    'family assignments made': status['family_assignments_made'],
    'network registration: held-out network files': network_registration['heldout_network_waveform_files_opened'],
    'network registration: held-out DAS files': network_registration['heldout_DAS_HDF5_files_opened'],
    'network registration: held-out catalog rows': network_registration['heldout_catalog_event_rows_opened'],
    'runner release: held-out network files': runner_release['heldout_network_waveform_files_opened_before_release'],
    'runner release: held-out DAS files': runner_release['heldout_DAS_HDF5_files_opened_before_release'],
    'runner release: held-out catalog rows': runner_release['heldout_catalog_event_rows_opened_before_release'],
    'runner release: held-out family rows': runner_release['heldout_family_label_rows_opened_before_release'],
}
assert all(value == 0 for value in access.values())
display(pd.Series({
    'registration status': registration['registration_status'],
    'v1 raw triggers': status['v1_raw_candidate_count'],
    'v2 retained triggers': status['v2_retained_candidate_count'],
    'v2 rejected triggers': status['v2_rejected_candidate_count'],
    'registered strong-block ratio': status['strong_block_characteristic_ratio'],
    'registered minimum strong blocks': status['minimum_strong_block_count'],
    'held-out hours still sealed': registration['heldout_total_duration_h'],
    'network registration status': network_registration['registration_status'],
    'network template threshold': network_registration['template_threshold'],
    'network generic threshold': network_registration['generic_threshold'],
    'historical template files pinned': network_registration['historical_template_available_waveform_count'],
    'runner release status': runner_release['status'],
    'runner/remote SHA': runner_release['runner_commit_sha'],
    'runner implementation files pinned': len(runner_release['runner_implementation_files']),
    'interpretation': status['interpretation'],
}, name='frozen checkpoint').to_frame())
display(pd.Series(access, name='count').to_frame())

## 2. What the single new gate does

Every v1 trigger already exceeded its interval-specific block-shift-null score threshold. Version 2 does not move that threshold. It requires broad support at the candidate peak: at least four of ten blocks must reach the existing ratio of 2. The separation below was seen during development, so its visual strength cannot substitute for held-out specificity.

In [ ]:
retained_parent_ids = set(v2['parent_v1_candidate_id'])
view = raw.copy()
view['v2_retained'] = view['candidate_id'].isin(retained_parent_ids)
assert view['v2_retained'].sum() == 2
assert set(view.loc[view['v2_retained'], 'candidate_id']) == retained_parent_ids

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), constrained_layout=True)
rejected = ~view['v2_retained']
axes[0].scatter(
    view.loc[rejected, 'block_support_count_at_declared_ratio'],
    view.loc[rejected, 'coincidence_score'],
    color='0.45', alpha=0.7, label='rejected by v2',
)
axes[0].scatter(
    view.loc[~rejected, 'block_support_count_at_declared_ratio'],
    view.loc[~rejected, 'coincidence_score'],
    marker='*', s=150, color='crimson', label='retained by v2', zorder=5,
)
axes[0].axvline(4, color='tab:blue', linestyle='--', label='frozen v2 gate')
axes[0].set(
    xlabel='blocks at characteristic ratio >= 2',
    ylabel='v1 fourth-highest block score',
    xticks=range(11),
    title='Development support separation',
)
axes[0].legend(loc='upper left')
axes[1].bar(
    ['v1 raw', 'v2 retained', 'v2 rejected'],
    [len(raw), len(v2), len(raw) - len(v2)],
    color=['0.55', 'tab:blue', '0.8'],
)
axes[1].set(ylabel='candidate count', title='Mechanical development replay')
for patch in axes[1].patches:
    axes[1].text(
        patch.get_x() + patch.get_width() / 2,
        patch.get_height() + 1,
        f'{int(patch.get_height())}',
        ha='center',
    )
plt.show()

display(v2[[
    'candidate_id',
    'parent_v1_candidate_id',
    'trigger_time',
    'coincidence_score',
    'strong_block_count',
    'v1_score_rank',
    'candidate_generation_label',
]])

## 3. Advisor sandbox (in memory only)

Change the values below and rerun this cell. This is a sensitivity display, not a new detector version. It never writes a product or changes the registered four-block gate. A different value cannot be selected after seeing held-out results.

In [ ]:
EXPLORATORY_MIN_STRONG_BLOCKS = 4
EXPLORATORY_MIN_V1_SCORE = None

mask = (
    view['block_support_count_at_declared_ratio']
    >= EXPLORATORY_MIN_STRONG_BLOCKS
)
if EXPLORATORY_MIN_V1_SCORE is not None:
    mask &= view['coincidence_score'] >= EXPLORATORY_MIN_V1_SCORE
sandbox = view.loc[mask].copy()
curve = pd.DataFrame({
    'minimum_strong_blocks': list(range(11)),
    'retained_v1_candidates': [
        int((view['block_support_count_at_declared_ratio'] >= count).sum())
        for count in range(11)
    ],
})
display(pd.Series({
    'exploratory minimum strong blocks': EXPLORATORY_MIN_STRONG_BLOCKS,
    'exploratory minimum v1 score': EXPLORATORY_MIN_V1_SCORE,
    'exploratory retained rows': len(sandbox),
    'frozen minimum strong blocks': status['minimum_strong_block_count'],
    'frozen retained rows': status['v2_retained_candidate_count'],
}, name='non-writing sandbox').to_frame())
display(curve)
display(sandbox[[
    'candidate_id', 'trigger_time', 'coincidence_score',
    'block_support_count_at_declared_ratio', 'v2_retained',
]].sort_values('coincidence_score', ascending=False))
assert sha256(V2 / 'candidate_detections.csv') == status['candidate_table_sha256']

## 4. The next scientific checkpoint

The DAS-v2 checkpoint and held-out network runner were verified on the private remote. The network run is now complete and its 33-row time-only union is frozen; see `06_heldout_network_freeze_checkpoint.ipynb` for the compact, advisor-playable result. The next gate is a registered catalog audit of that immutable union. Held-out DAS generation must remain independent and may not use network or catalog times.

A useful held-out result is not simply ‘some DAS triggers.’ It is an independently adjudicated local event increment beyond the full network union, or an improvement in held-out family-partition resolution, at the frozen event-level operating point.

In [ ]:
protocol = pd.DataFrame([
    {
        'order': 1,
        'stage': 'Freeze and push DAS-v2 implementation',
        'status': 'complete',
        'DAS held-out waveform access': 0,
    },
    {
        'order': 2,
        'stage': 'Register/release held-out network runner',
        'status': 'complete',
        'DAS held-out waveform access': 0,
    },
    {
        'order': 3,
        'stage': 'Run/freeze held-out network-only union',
        'status': 'complete; see notebook 06',
        'DAS held-out waveform access': 0,
    },
    {
        'order': 4,
        'stage': 'Register/audit immutable network union',
        'status': 'next',
        'DAS held-out waveform access': 0,
    },
    {
        'order': 5,
        'stage': 'Register/run frozen DAS-v2 independently',
        'status': 'pending',
        'DAS held-out waveform access': 'only through a new release gate',
    },
])
display(protocol.set_index('order'))
print('Runner release:', runner_release['status'])
print('Runner/remote SHA:', runner_release['runner_commit_sha'])
print('Historical registration catalog gate:', network_registration['heldout_catalog_access_gate'])
print('Historical registration DAS gate:', network_registration['heldout_DAS_access_gate'])
print('Current result and gates: open notebook 06.')